In [0]:
%sql
-- ============================================================
-- GOLD LAYER - UNITY CATALOG SETUP
-- ============================================================

-- 1. Create Gold External Location
CREATE EXTERNAL LOCATION IF NOT EXISTS gold_external_location
URL 'abfss://gold@utilitiesmaintainence.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL utilities_adls_credential);


-- 2. Create Gold Schema
CREATE SCHEMA IF NOT EXISTS motordatabricks.utilities_gold
MANAGED LOCATION 'abfss://gold@utilitiesmaintainence.dfs.core.windows.net/';


-- 3. Grant access to Gold External Location
GRANT READ FILES, WRITE FILES, CREATE EXTERNAL TABLE
ON EXTERNAL LOCATION gold_external_location
TO `1833017mdcs@cit.edu.in`;


-- 4. Grant access to Gold Schema
GRANT USE SCHEMA,CREATE TABLE
ON SCHEMA motordatabricks.utilities_gold
TO `1833017mdcs@cit.edu.in`;


-- 5. Grant Catalog access
GRANT USE CATALOG
ON CATALOG motordatabricks
TO `1833017mdcs@cit.edu.in`;

In [0]:
%sql

SELECT 'dim_asset' AS table_name, COUNT(*) AS row_count
FROM motordatabricks.utilities_gold.dim_asset

UNION ALL

SELECT 'dim_technician', COUNT(*)
FROM motordatabricks.utilities_gold.dim_technician

UNION ALL

SELECT 'dim_location', COUNT(*)
FROM motordatabricks.utilities_gold.dim_location

UNION ALL

SELECT 'dim_date', COUNT(*)
FROM motordatabricks.utilities_gold.dim_date

UNION ALL

SELECT 'dim_maintenance_type', COUNT(*)
FROM motordatabricks.utilities_gold.dim_maintenance_type

UNION ALL

SELECT 'dim_status', COUNT(*)
FROM motordatabricks.utilities_gold.dim_status

UNION ALL

SELECT 'fact_maintenance', COUNT(*)
FROM motordatabricks.utilities_gold.fact_maintenance;

table_name,row_count
dim_asset,500
dim_technician,100
dim_location,72
dim_date,188
dim_maintenance_type,660
dim_status,11
fact_maintenance,3600


In [0]:
%sql
-- ============================================================
-- CREATE EXTERNAL GOLD TABLES
-- Data will physically reside in ADLS Gold container
-- ============================================================


-- ============================================================
-- 1. DIM ASSET
-- ============================================================

CREATE TABLE motordatabricks.utilities_gold.dim_asset
(
    asset_key          BIGINT,
    asset_id           STRING,
    asset_type         STRING,
    manufacturer       STRING,
    model              STRING,
    serial_number      STRING,
    installation_date  DATE,
    location_id        STRING,
    city               STRING,
    region             STRING,
    criticality        STRING,
    asset_status       STRING
)
USING DELTA
LOCATION 'abfss://gold@utilitiesmaintainence.dfs.core.windows.net/dim_asset';


-- ============================================================
-- 2. DIM TECHNICIAN
-- ============================================================

CREATE TABLE motordatabricks.utilities_gold.dim_technician
(
    technician_key     BIGINT,
    technician_id      STRING,
    technician_name    STRING,
    skill_level        STRING,
    team               STRING,
    experience_years   INT
)
USING DELTA
LOCATION 'abfss://gold@utilitiesmaintainence.dfs.core.windows.net/dim_technician';


-- ============================================================
-- 3. DIM LOCATION
-- ============================================================

CREATE TABLE motordatabricks.utilities_gold.dim_location
(
    location_key       BIGINT,
    location_id        STRING,
    city               STRING,
    region             STRING
)
USING DELTA
LOCATION 'abfss://gold@utilitiesmaintainence.dfs.core.windows.net/dim_location';


-- ============================================================
-- 4. DIM DATE
-- ============================================================

CREATE TABLE motordatabricks.utilities_gold.dim_date
(
    date_key           INT,
    date               DATE,
    year               INT,
    quarter            INT,
    month              INT,
    month_name         STRING,
    week               INT,
    day                INT,
    day_of_week        INT,
    day_name           STRING,
    is_weekend         BOOLEAN
)
USING DELTA
LOCATION 'abfss://gold@utilitiesmaintainence.dfs.core.windows.net/dim_date';


-- ============================================================
-- 5. DIM MAINTENANCE TYPE
-- ============================================================

CREATE TABLE motordatabricks.utilities_gold.dim_maintenance_type
(
    maintenance_type_key   BIGINT,
    maintenance_type       STRING,
    maintenance_category   STRING,
    planned_flag            STRING,
    failure_type            STRING,
    failure_severity        STRING,
    priority                STRING
)
USING DELTA
LOCATION 'abfss://gold@utilitiesmaintainence.dfs.core.windows.net/dim_maintenance_type';


-- ============================================================
-- 6. DIM STATUS
-- ============================================================

CREATE TABLE motordatabricks.utilities_gold.dim_status
(
    status_key              BIGINT,
    work_order_status       STRING,
    resolution_code         STRING
)
USING DELTA
LOCATION 'abfss://gold@utilitiesmaintainence.dfs.core.windows.net/dim_status';


-- ============================================================
-- 7. FACT MAINTENANCE
-- ============================================================

CREATE TABLE motordatabricks.utilities_gold.fact_maintenance
(
    maintenance_key         BIGINT,
    work_order_id           STRING,

    asset_key               BIGINT,
    technician_key          BIGINT,
    location_key            BIGINT,
    maintenance_type_key    BIGINT,
    status_key              BIGINT,

    created_date_key        INT,
    scheduled_date_key      INT,
    completed_date_key      INT,

    downtime_hours          DOUBLE,
    labor_hours             DOUBLE,
    parts_used              INT,
    parts_cost              DOUBLE,
    labor_cost              DOUBLE,
    total_cost              DOUBLE,

    created_date            TIMESTAMP,
    scheduled_date          TIMESTAMP,
    completed_date          TIMESTAMP,

    source_system           STRING,
    updated_at              TIMESTAMP
)
USING DELTA
LOCATION 'abfss://gold@utilitiesmaintainence.dfs.core.windows.net/fact_maintenance';